In [9]:
import os
import torch
import torch.nn.functional as F
from pathlib import Path
from typing import List, Dict
import plotly.graph_objects as go
import pandas as pd
import numpy as np

1. compute norm(W[t+1]- W[t])-norm(W[t]-W[t-1]) for each checkpointing period (try with L2 norm, cosine distance)
2. compute [f[t+1]-f[t]] - [f[t]-f[t-1]]

## 1. compute matrix norm (frobenius and cosine distance)

### Useful Functions

In [2]:
def load_flat_tensor(path):
        return torch.load(path).flatten()

In [7]:
def get_filename(epoch: int, batch: int = None):
        if batch is not None:
            return f"{weight_type}_epoch_{epoch}_batch_{batch}.pt.pt"
        else:
            return f"{weight_type}_epoch_{epoch}.pt.pt"

In [5]:
checkpoints = (
        [(1, i) for i in range(0, 301)] +
        [(1, i) for i in range(400, 9300, 100)] +
        [(i, None) for i in range(1, 41)]
    )

In [16]:
def compute_weight_norm(checkpoint_directory: str, weight_type: str) -> Dict[str, List[float]]:
    
    checkpoint_dir = Path(checkpoint_directory)
    paths = [checkpoint_dir / get_filename(epoch, batch) for epoch, batch in checkpoints]
    paths = [p for p in paths if p.exists()]

    # Sanity check
    if len(paths) < 3:
        raise ValueError("Not enough checkpoints to compute second differences.")

    frob_diffs, cosine_diffs = [], []

    for i in range(1, len(paths) - 1):
        w_prev = load_flat_tensor(paths[i - 1])
        w_curr = load_flat_tensor(paths[i])
        w_next = load_flat_tensor(paths[i + 1])

        #second_diff = w_next - 2 * w_curr + w_prev
        delta_prev = w_curr - w_prev
        delta_next = w_next - w_curr

        frob = torch.norm(delta_next, p=2).item()-torch.norm(delta_prev, p=2).item()
        cosine = F.cosine_similarity(delta_prev, delta_next, dim=0).item()

        frob_diffs.append(frob)
        cosine_diffs.append(cosine)

    return {
        "frob": frob_diffs,
        "cosine": cosine_diffs
    }

In [25]:
def plot_breakthroughs(metrics: dict, checkpoints: list, title: str = "Breakthrough Analysis"):
    # Skip the first and last elements due to diff computation
    x_labels = checkpoints[1:-1]  # Format: (epoch, batch)

    # Format nicely for readability on the x-axis
    x_ticks = [f"e{e}_b{b}" for (e, b) in x_labels]

    fig = go.Figure()

    # Frobenius norm second difference
    fig.add_trace(go.Scatter(
        x=x_ticks,
        y=metrics["frob"],
        mode='lines+markers',
        name='Frobenius Norm',
        line=dict(color='royalblue')
    ))

    # Cosine distance second difference
    fig.add_trace(go.Scatter(
        x=x_ticks,
        y=metrics["cosine"],
        mode='lines+markers',
        name='Cosine Distance',
        line=dict(color='firebrick')
    ))

    fig.update_layout(
        title=title,
        xaxis_title='Checkpoint (Epoch_Batch)',
        yaxis_title='Breakthrough Magnitude',
        xaxis=dict(tickangle=45, tickmode='array', tickvals=x_ticks[::10], ticktext=x_ticks[::10]),
        legend=dict(
            x=1.02,
            y=1,
            traceorder="normal",
            bordercolor="Black",
            borderwidth=1,
            xanchor="left"
        ),
        template='plotly_white',
        width=1000,
        height=500
    )

    fig.show()


### Actual Plots

In [ ]:
weight_type = 'layer2_cell_gate_hh'
checkpoint_directory = '/scratch2/mrenaudin/colorlessgreenRNNs/checkpoints/lstm_adam_full_check_shuffled/weights'
#300 first batches
checkpoints = [(1, i) for i in range(0, 301)]
l2_cell_hh_300batches = compute_weight_norm(checkpoint_directory, weight_type)
plot_breakthroughs(l2_cell_hh_300batches, checkpoints, title = 'Breakthrough Analysis of the Recurrent Weight of Cell State in the 2nd Layer (300 first batches)')

#Remaining of the first epoch
checkpoints = [(1, i) for i in range(400, 9300, 100)]
l2_cell_hh_1st_ep = compute_weight_norm(checkpoint_directory, weight_type)
plot_breakthroughs(l2_cell_hh_1st_ep, checkpoints, title = 'Breakthrough Analysis of the Recurrent Weight of Cell State in the 2nd Layer (Remaining of the first epoch)')

#Full training
checkpoints = [(i, None) for i in range(1, 41)]
l2_cell_40_ep = compute_weight_norm(checkpoint_directory, weight_type)
plot_breakthroughs(l2_cell_40_ep, checkpoints, title = 'Breakthrough Analysis of the Recurrent Weight of Cell State in the 2nd Layer (40 epochs)')


In [ ]:
weight_type = 'layer2_forget_gate_hh'
checkpoint_directory = '/scratch2/mrenaudin/colorlessgreenRNNs/checkpoints/lstm_adam_full_check_shuffled/weights'
#300 first batches
checkpoints = [(1, i) for i in range(0, 301)]
l2_forget_hh_300batches = compute_weight_norm(checkpoint_directory, weight_type)
plot_breakthroughs(l2_forget_hh_300batches, checkpoints, title = 'Breakthrough Analysis of the Recurrent Weight of Forget Gate in the 2nd Layer (300 first batches)')

#Remaining of the first epoch
checkpoints = [(1, i) for i in range(400, 9300, 100)]
l2_forget_hh_1st_ep = compute_weight_norm(checkpoint_directory, weight_type)
plot_breakthroughs(l2_forget_hh_1st_ep, checkpoints, title = 'Breakthrough Analysis of the Recurrent Weight of Forget Gate in the 2nd Layer (Remaining of the first epoch)')

#Full training
checkpoints = [(i, None) for i in range(1, 41)]
l2_forget_40_ep = compute_weight_norm(checkpoint_directory, weight_type)
plot_breakthroughs(l2_forget_40_ep, checkpoints, title = 'Breakthrough Analysis of the Recurrent Weight of Forget Gate in the 2nd Layer (40 epochs)')


In [ ]:
weight_type = 'layer2_input_gate_hh'
checkpoint_directory = '/scratch2/mrenaudin/colorlessgreenRNNs/checkpoints/lstm_adam_full_check_shuffled/weights'
#300 first batches
checkpoints = [(1, i) for i in range(0, 301)]
l2_input_hh_300batches = compute_weight_norm(checkpoint_directory, weight_type)
plot_breakthroughs(l2_input_hh_300batches, checkpoints, title = 'Breakthrough Analysis of the Recurrent Weight of Input Gate in the 2nd Layer (300 first batches)')

#Remaining of the first epoch
checkpoints = [(1, i) for i in range(400, 9300, 100)]
l2_input_hh_1st_ep = compute_weight_norm(checkpoint_directory, weight_type)
plot_breakthroughs(l2_input_hh_1st_ep, checkpoints, title = 'Breakthrough Analysis of the Recurrent Weight of Input Gate in the 2nd Layer (Remaining of the first epoch)')

#Full training
checkpoints = [(i, None) for i in range(1, 41)]
l2_input_40_ep = compute_weight_norm(checkpoint_directory, weight_type)
plot_breakthroughs(l2_input_40_ep, checkpoints, title = 'Breakthrough Analysis of the Recurrent Weight of Input Gate in the 2nd Layer (40 epochs)')


In [ ]:
weight_type = 'layer2_output_gate_hh'
checkpoint_directory = '/scratch2/mrenaudin/colorlessgreenRNNs/checkpoints/lstm_adam_full_check_shuffled/weights'
#300 first batches
checkpoints = [(1, i) for i in range(0, 301)]
l2_output_hh_300batches = compute_weight_norm(checkpoint_directory, weight_type)
plot_breakthroughs(l2_output_hh_300batches, checkpoints, title = 'Breakthrough Analysis of the Recurrent Weight of Output Gate in the 2nd Layer (300 first batches)')

#Remaining of the first epoch
checkpoints = [(1, i) for i in range(400, 9300, 100)]
l2_output_hh_1st_ep = compute_weight_norm(checkpoint_directory, weight_type)
plot_breakthroughs(l2_output_hh_1st_ep, checkpoints, title = 'Breakthrough Analysis of the Recurrent Weight of Output Gate in the 2nd Layer (Remaining of the first epoch)')

#Full training
checkpoints = [(i, None) for i in range(1, 41)]
l2_output_40_ep = compute_weight_norm(checkpoint_directory, weight_type)
plot_breakthroughs(l2_output_40_ep, checkpoints, title = 'Breakthrough Analysis of the Recurrent Weight of Output Gate in the 2nd Layer (40 epochs)')


## 2. Compute breakthrough on Nounpp conditions

In [2]:
df = pd.read_csv('/scratch2/mrenaudin/colorlessgreenRNNs/evaluation_notebooks/results/lstm_adam_full_check_shuffled')

In [ ]:
df

In [ ]:
checkpoints = (
        [(1, i) for i in range(0, 301)] +
        [(1, i) for i in range(400, 9300, 100)] +
        [(i, None) for i in range(1, 41)]
    )

In [ ]:
df['checkpoints']= checkpoints

In [ ]:
df

In [ ]:


#300 first batches
valid_checkpoints = [(1, i) for i in range(301)]
#filter df
df_filtered = df[df['checkpoints'].isin(valid_checkpoints)].copy()

# # Sort by checkpoint second element to ensure order
# df_filtered['checkpoint_idx'] = df_filtered['checkpoints'].apply(lambda x: x[1])
# df_filtered = df_filtered.sort_values('checkpoint_idx').reset_index(drop=True)

# Conditions to calculate for
conditions = ['singular singular', 'singular plural', 'plural singular', 'plural plural']

# Initialize dictionary to store results
second_diff_results = {cond: [] for cond in conditions}

# Compute second difference for indices 1 to len-2 (since we need t-1 and t+1)
for i in range(1, len(df_filtered) - 1):
    for cond in conditions:
        f_t_plus_1 = df_filtered.loc[i+1, cond]
        f_t = df_filtered.loc[i, cond]
        f_t_minus_1 = df_filtered.loc[i-1, cond]
        second_diff = f_t_plus_1 - 2*f_t + f_t_minus_1
        second_diff_results[cond].append(second_diff)

# For example, create a new DataFrame for second differences (length 299 since edges are excluded)
second_diff_df = pd.DataFrame(second_diff_results)
#second_diff_df['checkpoint_idx'] = df_filtered.loc[1:-1, 'checkpoint_idx'].values

print(second_diff_df)

    
    

In [ ]:

fig = go.Figure()

conditions = ['singular singular', 'singular plural', 'plural singular', 'plural plural']

# x-axis: just use the index of the dataframe rows, which corresponds to i = 1..299
x_vals = second_diff_df.index

for cond in conditions:
    fig.add_trace(go.Scatter(
        x=x_vals,
        y=second_diff_df[cond],
        mode='lines+markers',
        name=cond
    ))

fig.update_layout(
    title='Second Difference of Conditions over Checkpoints',
    xaxis_title='Checkpoint index (i)',
    yaxis_title='Second difference',
    legend_title='Condition',
    template='plotly_white'
)

fig.show()

